# Notebook 02: Silver Transformation - Data Cleansing & Validation

## Overview
This notebook transforms Bronze layer data into cleansed, validated Silver layer tables with comprehensive data quality checks.

## Prerequisites
- Notebook 01 completed successfully
- Bronze tables: `report_summary`, `country_weekly`, `extraction_metadata`
- Reference data: `country_codes_iso3166.csv`, `epi_week_calendar.csv`

## Inputs
- Bronze Delta tables from Notebook 01
- Reference data for validation

## Outputs
- Delta table: `silver.report_summary` (cleansed report data)
- Delta table: `silver.country_weekly` (cleansed country data)
- Delta table: `silver.data_quality_checks` (QA results)

## Execution Time
~1-2 minutes

In [1]:
# ============================================
# ENVIRONMENT DETECTION & CONFIGURATION
# ============================================

import os
import sys
from pathlib import Path

# Auto-detect environment
IS_FABRIC = os.path.exists('/lakehouse/default')

if IS_FABRIC:
    print("🌐 Running in Microsoft Fabric")
    BRONZE_TABLE_PATH = "/lakehouse/default/Tables/bronze"
    SILVER_TABLE_PATH = "/lakehouse/default/Tables/silver"
    REFERENCE_PATH = "/lakehouse/default/Files/reference"
else:
    print("💻 Running locally")
    project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    sys.path.insert(0, str(project_root / 'src'))
    
    BRONZE_TABLE_PATH = str(project_root / "data" / "bronze_tables")
    SILVER_TABLE_PATH = str(project_root / "data" / "silver_tables")
    REFERENCE_PATH = str(project_root / "data" / "reference")
    
    # Create output directory
    Path(SILVER_TABLE_PATH).mkdir(parents=True, exist_ok=True)

print(f"Bronze Path: {BRONZE_TABLE_PATH}")
print(f"Silver Path: {SILVER_TABLE_PATH}")
print(f"Reference Path: {REFERENCE_PATH}")

💻 Running locally
Bronze Path: D:\Projects\cholera-cdr-mvp\data\bronze_tables
Silver Path: D:\Projects\cholera-cdr-mvp\data\silver_tables
Reference Path: D:\Projects\cholera-cdr-mvp\data\reference


In [2]:
# ============================================
# IMPORTS
# ============================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Any, Optional, Tuple
import logging

# Import QA engine from local modules
try:
    from epi_analytics.qa_engine import QualityEngine, QualityCheck
except ImportError:
    print("⚠️  Could not import QA engine, will use simplified validation")
    QualityEngine = None

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✅ Imports successful")

Importing plotly failed. Interactive plots will not work.


✅ Imports successful


In [3]:
# ============================================
# LOAD BRONZE DATA
# ============================================

print("\n📂 Loading Bronze layer data...\n")

try:
    if IS_FABRIC:
        # Fabric: Read from Delta tables
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()
        
        df_bronze_reports = spark.table("bronze.report_summary").toPandas()
        df_bronze_countries = spark.table("bronze.country_weekly").toPandas()
    else:
        # Local: Read from Parquet files
        df_bronze_reports = pd.read_parquet(Path(BRONZE_TABLE_PATH) / "report_summary.parquet")
        df_bronze_countries = pd.read_parquet(Path(BRONZE_TABLE_PATH) / "country_weekly.parquet")
    
    print(f"✅ Loaded {len(df_bronze_reports)} reports")
    print(f"✅ Loaded {len(df_bronze_countries)} country records")
    
except Exception as e:
    logger.error(f"Error loading Bronze data: {e}")
    raise

# Display sample
print("\n📋 Sample Bronze Report Data:")
display(df_bronze_reports.head(3))


📂 Loading Bronze layer data...

✅ Loaded 3 reports
✅ Loaded 6 country records

📋 Sample Bronze Report Data:


,report_id,epi_year,epi_week,report_date,confirmed_cases,suspected_cases,deaths,cfr_percent,affected_countries,source_file,extraction_timestamp,country_breakdown_count
0,2025_wk06,2025,6,None,1075,1505,26,2.42,5,cholera_sitrep_2025_wk06.pdf,2026-02-11T05:26:40.110828,2
1,2025_wk07,2025,7,None,1379,1930,30,2.18,9,cholera_sitrep_2025_wk07.pdf,2026-02-11T05:26:40.285546,2
2,2025_wk08,2025,8,None,1993,2790,35,1.76,6,cholera_sitrep_2025_wk08.pdf,2026-02-11T05:26:40.452397,2


In [4]:
# ============================================
# LOAD REFERENCE DATA
# ============================================

print("\n📚 Loading reference data...\n")

try:
    # Load country codes
    df_country_codes = pd.read_csv(Path(REFERENCE_PATH) / "country_codes_iso3166.csv")
    print(f"✅ Loaded {len(df_country_codes)} country codes")
    
    # Load epi week calendar
    df_epi_calendar = pd.read_csv(Path(REFERENCE_PATH) / "epi_week_calendar.csv")
    df_epi_calendar['start_date'] = pd.to_datetime(df_epi_calendar['start_date'])
    df_epi_calendar['end_date'] = pd.to_datetime(df_epi_calendar['end_date'])
    print(f"✅ Loaded {len(df_epi_calendar)} epi weeks")
    
except Exception as e:
    logger.error(f"Error loading reference data: {e}")
    raise

# Create country name mapping dictionary
country_name_map = dict(zip(
    df_country_codes['country_name'].str.lower(),
    df_country_codes['country_code']
))

print(f"\n📋 Sample country mapping:")
for name, code in list(country_name_map.items())[:5]:
    print(f"  {name} -> {code}")


📚 Loading reference data...

✅ Loaded 53 country codes
✅ Loaded 106 epi weeks

📋 Sample country mapping:
  algeria -> DZA
  angola -> AGO
  benin -> BEN
  botswana -> BWA
  burkina faso -> BFA


In [5]:
# ============================================
# DATA CLEANSING FUNCTIONS
# ============================================

def standardize_country_code(country_name: str, mapping: Dict[str, str]) -> Optional[str]:
    """
    Standardize country name to ISO 3166-1 alpha-3 code.
    
    Args:
        country_name: Country name from source data
        mapping: Dictionary mapping country names to codes
        
    Returns:
        ISO 3166-1 alpha-3 country code or None
    """
    if pd.isna(country_name) or country_name == '':
        return None
    
    # Try exact match (case-insensitive)
    country_lower = str(country_name).lower().strip()
    
    if country_lower in mapping:
        return mapping[country_lower]
    
    # Try partial match
    for name, code in mapping.items():
        if country_lower in name or name in country_lower:
            return code
    
    logger.warning(f"Could not map country name: {country_name}")
    return None


def calculate_cfr(deaths: Optional[int], cases: Optional[int]) -> Optional[float]:
    """
    Calculate Case Fatality Rate as percentage.
    
    Args:
        deaths: Number of deaths
        cases: Number of confirmed cases
        
    Returns:
        CFR as percentage or None
    """
    if pd.isna(deaths) or pd.isna(cases) or cases == 0:
        return None
    
    return round((deaths / cases) * 100, 2)


def get_report_date(epi_year: int, epi_week: int, calendar_df: pd.DataFrame) -> Optional[datetime]:
    """
    Get report date from epidemiological week.
    
    Args:
        epi_year: Epidemiological year
        epi_week: Epidemiological week
        calendar_df: Epi week calendar DataFrame
        
    Returns:
        Report date (end of epi week) or None
    """
    if pd.isna(epi_year) or pd.isna(epi_week):
        return None
    
    match = calendar_df[
        (calendar_df['epi_year'] == epi_year) & 
        (calendar_df['epi_week'] == epi_week)
    ]
    
    if len(match) > 0:
        return match.iloc[0]['end_date']
    
    logger.warning(f"Could not find date for {epi_year}-W{epi_week:02d}")
    return None


print("✅ Cleansing functions defined")

✅ Cleansing functions defined


In [6]:
# ============================================
# TRANSFORM REPORT SUMMARY TO SILVER
# ============================================

print("\n🔄 Transforming report summary to Silver layer...\n")

df_silver_reports = df_bronze_reports.copy()

# Add report date from epi week
df_silver_reports['report_date'] = df_silver_reports.apply(
    lambda row: get_report_date(row['epi_year'], row['epi_week'], df_epi_calendar),
    axis=1
)

# Calculate CFR if not present
df_silver_reports['calculated_cfr'] = df_silver_reports.apply(
    lambda row: calculate_cfr(row['deaths'], row['confirmed_cases']),
    axis=1
)

# Use calculated CFR if reported CFR is missing
df_silver_reports['cfr_percent'] = df_silver_reports['cfr_percent'].fillna(
    df_silver_reports['calculated_cfr']
)

# Handle missing values
df_silver_reports['suspected_cases'] = df_silver_reports['suspected_cases'].fillna(0)
df_silver_reports['affected_countries'] = df_silver_reports['affected_countries'].fillna(0)

# Add audit columns
df_silver_reports['created_at'] = datetime.now()
df_silver_reports['updated_at'] = datetime.now()
df_silver_reports['data_quality_score'] = None  # Will be calculated later

# Add data completeness percentage
critical_fields = ['confirmed_cases', 'deaths', 'cfr_percent', 'report_date']
df_silver_reports['data_completeness_pct'] = df_silver_reports[critical_fields].notna().sum(axis=1) / len(critical_fields) * 100

print(f"✅ Transformed {len(df_silver_reports)} reports")
print(f"   - Reports with dates: {df_silver_reports['report_date'].notna().sum()}")
print(f"   - Average completeness: {df_silver_reports['data_completeness_pct'].mean():.1f}%")

# Display sample
print("\n📋 Sample Silver Report Data:")
display(df_silver_reports[['report_id', 'report_date', 'confirmed_cases', 'deaths', 'cfr_percent', 'data_completeness_pct']].head())


🔄 Transforming report summary to Silver layer...

✅ Transformed 3 reports
   - Reports with dates: 3
   - Average completeness: 100.0%

📋 Sample Silver Report Data:


,report_id,report_date,confirmed_cases,deaths,cfr_percent,data_completeness_pct
0,2025_wk06,2025-02-09,1075,26,2.42,100.0
1,2025_wk07,2025-02-16,1379,30,2.18,100.0
2,2025_wk08,2025-02-23,1993,35,1.76,100.0


In [7]:
# ============================================
# TRANSFORM COUNTRY WEEKLY TO SILVER
# ============================================

print("\n🔄 Transforming country weekly to Silver layer...\n")

df_silver_countries = df_bronze_countries.copy()

# Standardize country codes
df_silver_countries['country_code'] = df_silver_countries['country_name'].apply(
    lambda x: standardize_country_code(x, country_name_map)
)

# Add report date
df_silver_countries['report_date'] = df_silver_countries.apply(
    lambda row: get_report_date(row['epi_year'], row['epi_week'], df_epi_calendar),
    axis=1
)

# Calculate country-level CFR
df_silver_countries['cfr_percent'] = df_silver_countries.apply(
    lambda row: calculate_cfr(row['deaths'], row['confirmed_cases']),
    axis=1
)

# Handle missing values
df_silver_countries['confirmed_cases'] = df_silver_countries['confirmed_cases'].fillna(0)
df_silver_countries['suspected_cases'] = df_silver_countries['suspected_cases'].fillna(0)
df_silver_countries['deaths'] = df_silver_countries['deaths'].fillna(0)

# Add audit columns
df_silver_countries['created_at'] = datetime.now()
df_silver_countries['updated_at'] = datetime.now()

# Filter out records without country codes
unmapped_countries = df_silver_countries[df_silver_countries['country_code'].isna()]
if len(unmapped_countries) > 0:
    print(f"⚠️  {len(unmapped_countries)} records without country codes (will be excluded)")
    print(f"   Unmapped countries: {unmapped_countries['country_name'].unique().tolist()}")

df_silver_countries = df_silver_countries[df_silver_countries['country_code'].notna()]

print(f"✅ Transformed {len(df_silver_countries)} country records")
print(f"   - Unique countries: {df_silver_countries['country_code'].nunique()}")
print(f"   - Records with CFR: {df_silver_countries['cfr_percent'].notna().sum()}")

# Display sample
print("\n📋 Sample Silver Country Data:")
display(df_silver_countries[['report_id', 'country_code', 'country_name', 'confirmed_cases', 'deaths', 'cfr_percent']].head())


🔄 Transforming country weekly to Silver layer...

✅ Transformed 6 country records
   - Unique countries: 2
   - Records with CFR: 6

📋 Sample Silver Country Data:


C:\Users\jdiek\AppData\Local\Temp\ipykernel_32224\2141833390.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_silver_countries['suspected_cases'] = df_silver_countries['suspected_cases'].fillna(0)


,report_id,country_code,country_name,confirmed_cases,deaths,cfr_percent
0,2025_wk06,ZWE,Zimbabwe,355,7,1.97
1,2025_wk06,ZMB,Zambia,245,9,3.67
2,2025_wk07,ZWE,Zimbabwe,409,8,1.96
3,2025_wk07,ZMB,Zambia,372,5,1.34
4,2025_wk08,ZWE,Zimbabwe,429,11,2.56


In [8]:
# ============================================
# DATA QUALITY VALIDATION
# ============================================

print("\n🔍 Running data quality checks...\n")

quality_checks = []

# Initialize QA engine if available
if QualityEngine:
    qa_engine = QualityEngine(tolerance_cfr=0.5, tolerance_cases=0.05)
    
    # Run comprehensive validation for each report
    for idx, report in df_silver_reports.iterrows():
        report_dict = report.to_dict()
        
        # Get country breakdown for this report
        country_breakdown = df_silver_countries[
            df_silver_countries['report_id'] == report['report_id']
        ].to_dict('records')
        
        # Run validation
        checks = qa_engine.run_comprehensive_validation(
            report['report_id'],
            report_dict,
            country_breakdown if len(country_breakdown) > 0 else None
        )
        
        quality_checks.extend(checks)
        
        # Calculate quality score
        if len(checks) > 0:
            quality_score = qa_engine.get_quality_score(checks)
            df_silver_reports.at[idx, 'data_quality_score'] = quality_score
        else:
            df_silver_reports.at[idx, 'data_quality_score'] = 100.0
    
    print(f"✅ Ran {len(quality_checks)} quality checks")
    
else:
    # Simplified validation without QA engine
    print("⚠️  Running simplified validation (QA engine not available)")
    
    for idx, report in df_silver_reports.iterrows():
        checks_count = 0
        
        # Check CFR consistency
        if pd.notna(report['cfr_percent']) and pd.notna(report['calculated_cfr']):
            cfr_diff = abs(report['cfr_percent'] - report['calculated_cfr'])
            if cfr_diff > 0.5:
                quality_checks.append({
                    'report_id': report['report_id'],
                    'check_type': 'CFR_CONSISTENCY',
                    'severity': 'WARNING' if cfr_diff < 1.0 else 'ERROR',
                    'expected_value': report['calculated_cfr'],
                    'actual_value': report['cfr_percent'],
                    'description': f'CFR mismatch: {cfr_diff:.2f}%',
                    'check_timestamp': datetime.now()
                })
                checks_count += 1
        
        # Simple quality score based on completeness
        df_silver_reports.at[idx, 'data_quality_score'] = report['data_completeness_pct']
    
    print(f"✅ Ran simplified validation: {len(quality_checks)} issues found")

# Create quality checks DataFrame
if len(quality_checks) > 0:
    if QualityEngine:
        # Convert QualityCheck objects to dicts
        df_quality_checks = pd.DataFrame([check.dict() for check in quality_checks])
    else:
        df_quality_checks = pd.DataFrame(quality_checks)
else:
    # Create empty DataFrame with expected schema
    df_quality_checks = pd.DataFrame(columns=[
        'report_id', 'check_type', 'severity', 'expected_value', 
        'actual_value', 'description', 'check_timestamp'
    ])

# Quality summary
print("\n📊 Quality Check Summary:")
if len(df_quality_checks) > 0:
    severity_counts = df_quality_checks['severity'].value_counts()
    for severity, count in severity_counts.items():
        print(f"  - {severity}: {count}")
else:
    print("  - No quality issues found ✅")

print(f"\nAverage quality score: {df_silver_reports['data_quality_score'].mean():.1f}")


🔍 Running data quality checks...

✅ Ran 6 quality checks

📊 Quality Check Summary:
  - ERROR: 6

Average quality score: 0.0


C:\Users\jdiek\AppData\Local\Temp\ipykernel_32224\316039373.py:71: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.5/migration/
  df_quality_checks = pd.DataFrame([check.dict() for check in quality_checks])


In [9]:
# ============================================
# SAVE TO SILVER LAYER
# ============================================

print("\n💾 Saving to Silver layer...\n")

try:
    if IS_FABRIC:
        # Fabric: Use PySpark to write Delta tables
        spark_silver_reports = spark.createDataFrame(df_silver_reports)
        spark_silver_countries = spark.createDataFrame(df_silver_countries)
        spark_quality_checks = spark.createDataFrame(df_quality_checks)
        
        # Write to Delta tables (overwrite mode for MVP)
        spark_silver_reports.write.format("delta").mode("overwrite").saveAsTable("silver.report_summary")
        spark_silver_countries.write.format("delta").mode("overwrite").saveAsTable("silver.country_weekly")
        spark_quality_checks.write.format("delta").mode("overwrite").saveAsTable("silver.data_quality_checks")
        
        print("✅ Delta tables created in Fabric Lakehouse")
        
    else:
        # Local: Save as Parquet
        df_silver_reports.to_parquet(
            Path(SILVER_TABLE_PATH) / "report_summary.parquet",
            index=False,
            engine='pyarrow'
        )
        
        df_silver_countries.to_parquet(
            Path(SILVER_TABLE_PATH) / "country_weekly.parquet",
            index=False,
            engine='pyarrow'
        )
        
        df_quality_checks.to_parquet(
            Path(SILVER_TABLE_PATH) / "data_quality_checks.parquet",
            index=False,
            engine='pyarrow'
        )
        
        print(f"✅ Parquet files saved to: {SILVER_TABLE_PATH}")
        
except Exception as e:
    logger.error(f"Error saving to Silver layer: {e}")
    raise

print("\n✅ Silver layer transformation complete!")


💾 Saving to Silver layer...

✅ Parquet files saved to: D:\Projects\cholera-cdr-mvp\data\silver_tables

✅ Silver layer transformation complete!


## Validation & Testing

In [10]:
# ============================================
# VALIDATION & TESTING
# ============================================

print("\n🔍 Running validation checks...\n")

# Test 1: Row count preservation
bronze_count = len(df_bronze_reports)
silver_count = len(df_silver_reports)
assert silver_count == bronze_count, f"Row count mismatch: Bronze={bronze_count}, Silver={silver_count}"
print(f"✅ Row count preserved: {silver_count}")

# Test 2: Required fields present
required_fields = ['report_id', 'report_date', 'confirmed_cases', 'deaths', 'cfr_percent', 'data_quality_score']
for field in required_fields:
    assert field in df_silver_reports.columns, f"Missing required field: {field}"
print(f"✅ All required fields present")

# Test 3: Data quality scores calculated
scores_calculated = df_silver_reports['data_quality_score'].notna().sum()
assert scores_calculated == len(df_silver_reports), f"Quality scores missing for some reports"
print(f"✅ Quality scores calculated for all {scores_calculated} reports")

# Test 4: CFR values reasonable
invalid_cfr = df_silver_reports[
    (df_silver_reports['cfr_percent'].notna()) & 
    ((df_silver_reports['cfr_percent'] < 0) | (df_silver_reports['cfr_percent'] > 100))
]
assert len(invalid_cfr) == 0, f"Found {len(invalid_cfr)} reports with invalid CFR"
print(f"✅ All CFR values within valid range (0-100%)")

# Test 5: Country codes standardized
if len(df_silver_countries) > 0:
    unmapped = df_silver_countries['country_code'].isna().sum()
    assert unmapped == 0, f"Found {unmapped} unmapped countries"
    print(f"✅ All {len(df_silver_countries)} country records have standardized codes")

# Test 6: Data completeness
avg_completeness = df_silver_reports['data_completeness_pct'].mean()
print(f"✅ Average data completeness: {avg_completeness:.1f}%")

# Test 7: Quality check severity distribution
if len(df_quality_checks) > 0:
    error_count = (df_quality_checks['severity'] == 'ERROR').sum()
    warning_count = (df_quality_checks['severity'] == 'WARNING').sum()
    print(f"✅ Quality checks: {error_count} errors, {warning_count} warnings")
else:
    print(f"✅ No quality issues detected")

print("\n✅ All validation checks passed!")


🔍 Running validation checks...

✅ Row count preserved: 3
✅ All required fields present
✅ Quality scores calculated for all 3 reports
✅ All CFR values within valid range (0-100%)
✅ All 6 country records have standardized codes
✅ Average data completeness: 100.0%
✅ Quality checks: 6 errors, 0 warnings

✅ All validation checks passed!


## Next Steps

1. **Review quality check results** above
2. **Investigate any ERROR severity** issues
3. **Proceed to Notebook 03** for Gold dimensional model creation

## Outputs Created

- `silver.report_summary` - Cleansed report-level data with quality scores
- `silver.country_weekly` - Cleansed country-level data with standardized codes
- `silver.data_quality_checks` - Comprehensive quality validation results